# MiohRestorer V5 — Google Colab CUDA training

This notebook creates resumable PyTorch/EMA checkpoints for **Motion-Informed Optical Healing V5** on an NVIDIA GPU. It preserves the native 128/192/256/384/512 ROI buckets and never resizes a crop to another bucket.

Current fail-closed scope: the repository implements the paired clean-GT losses needed by V5 Stage 3 (faithful structure) and Stage 4 (detail recovery). Stages 1, 2, 5 and 6 still require their exact-motion, natural-flow, feature-distillation or flow-aligned temporal teachers. The training command rejects those stages instead of silently producing a model with missing losses.

Colab produces `.pth` checkpoints. Core AI conversion, `.aimodelc` compilation and ANE validation remain Mac/Xcode 27 tasks.

## 1. Runtime and project settings

Select **Runtime → Change runtime type → NVIDIA GPU** first. For V5-Q at 384/512, an A100 40 GB is recommended. Keep checkpoints on Drive because Colab sessions can end without warning.

In [ ]:
from pathlib import Path

# Source: upload a zip containing the lada repository to Drive.
DRIVE_PROJECT = Path('/content/drive/MyDrive/mioh-v5')
REPO_ARCHIVE = DRIVE_PROJECT / 'lada-source.zip'
REPO_URL = ''  # Optional public Git URL. Leave empty when using REPO_ARCHIVE.
REPO_DIR = Path('/content/lada')

# Data directory must contain the manifests and every referenced target/mask video.
DRIVE_DATA_DIR = DRIVE_PROJECT / 'data'
LOCAL_DATA_DIR = Path('/content/mioh-v5-data')
STAGE_DATA_TO_LOCAL = True  # Faster than decoding videos directly from Drive.
TRAIN_MANIFEST_NAME = 'train-native.jsonl'
VALIDATION_MANIFEST_NAME = 'validation-native.jsonl'

# Training defaults: start with a short Stage 3 pilot.
VARIANT = 'q'       # 'q' = quality reference, 's' = shipping model
STAGE = 3           # This runner currently accepts 3 or 4 only
STEPS = 500         # Set 15000 for full Stage 3 after the pilot passes
BATCH_SIZE = 1
ACCUMULATE = 4      # Effective batch = BATCH_SIZE × ACCUMULATE
WORKERS = 2
SAVE_EVERY = 500
VALIDATE_EVERY = 500
VALIDATION_BATCHES = 24
RUN_NAME = f'v5-{VARIANT}-stage{STAGE}'
WORK_DIR = DRIVE_PROJECT / 'runs' / RUN_NAME

DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
print('Drive project:', DRIVE_PROJECT)
print('Run:', RUN_NAME, 'steps:', STEPS)

In [ ]:
import os, platform, subprocess, sys
import torch

assert torch.cuda.is_available(), 'GPU runtime is not enabled in Colab'
gpu = torch.cuda.get_device_properties(0)
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', gpu.name)
print('VRAM GiB:', round(gpu.total_memory / 2**30, 2))
print('bf16:', torch.cuda.is_bf16_supported())

## 2. Mount Drive and prepare the source tree

The notebook never stores a GitHub token. For a private repository, uploading `lada-source.zip` to the configured Drive path is the simplest option. The archive should contain a top-level `lada` directory or the repository files directly.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

if not (REPO_DIR / 'pyproject.toml').is_file():
    if REPO_ARCHIVE.is_file():
        unpack_root = Path('/content/mioh-source-unpacked')
        shutil.rmtree(unpack_root, ignore_errors=True)
        shutil.unpack_archive(str(REPO_ARCHIVE), str(unpack_root))
        candidates = [unpack_root] + [p for p in unpack_root.iterdir() if p.is_dir()]
        source = next((p for p in candidates if (p / 'pyproject.toml').is_file()), None)
        if source is None:
            raise FileNotFoundError('pyproject.toml was not found in lada-source.zip')
        shutil.copytree(source, REPO_DIR, dirs_exist_ok=True)
    elif REPO_URL:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        raise FileNotFoundError(f'Upload {REPO_ARCHIVE} or set REPO_URL')

os.chdir(REPO_DIR)
print('Repository:', REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip() if (REPO_DIR / '.git').exists() else 'archive source')

## 3. Install only the Linux/CUDA dependencies

Colab already supplies CUDA PyTorch. This deliberately does not install Core ML, Core AI, MLX, PyObjC or the macOS GUI dependencies.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'av>=16.1.0', 'opencv-python-headless==4.12.0.88',
    'mmengine==0.10.7', 'ultralytics==8.4.4', 'wcwidth', 'tqdm'
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

from lada.models.mioh_restorer.model_v5 import MiohRestorerV5, parameter_count
print('V5-Q parameters:', f'{parameter_count(MiohRestorerV5.quality()):,}')
print('V5-S parameters:', f'{parameter_count(MiohRestorerV5.shipping()):,}')

## 4. Stage the native dataset

Place `train-native.jsonl`, `validation-native.jsonl`, and their referenced videos under `DRIVE_DATA_DIR`. Relative paths are portable automatically. If an old manifest contains absolute Mac paths, set the two prefix variables below and the cell will rewrite only the copied manifests.

In [ ]:
import json

if STAGE_DATA_TO_LOCAL:
    if not DRIVE_DATA_DIR.is_dir():
        raise FileNotFoundError(DRIVE_DATA_DIR)
    shutil.rmtree(LOCAL_DATA_DIR, ignore_errors=True)
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    DATA_DIR = LOCAL_DATA_DIR
else:
    DATA_DIR = DRIVE_DATA_DIR

TRAIN_MANIFEST = DATA_DIR / TRAIN_MANIFEST_NAME
VALIDATION_MANIFEST = DATA_DIR / VALIDATION_MANIFEST_NAME
MAC_PATH_PREFIX = ''      # Example: /Volumes/Project_HD/mioh-v5-data
COLAB_PATH_PREFIX = str(DATA_DIR)

def rewrite_manifest_prefix(path: Path, old: str, new: str) -> None:
    if not old:
        return
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        for key in ('target_video', 'mask_video'):
            value = str(row[key])
            if value.startswith(old):
                row[key] = new + value[len(old):]
        rows.append(row)
    path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')

for manifest in (TRAIN_MANIFEST, VALIDATION_MANIFEST):
    if not manifest.is_file():
        raise FileNotFoundError(manifest)
    rewrite_manifest_prefix(manifest, MAC_PATH_PREFIX, COLAB_PATH_PREFIX)
print('Train manifest:', TRAIN_MANIFEST)
print('Validation manifest:', VALIDATION_MANIFEST)

In [ ]:
from collections import Counter
from lada.models.mioh_restorer.native_dataset_v5 import MiohRestorerV5NativeDataset

def inspect_manifest(path: Path):
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    counts = Counter(int(row['bucket']) for row in rows)
    missing = []
    for row in rows:
        for key in ('target_video', 'mask_video'):
            value = Path(str(row[key]))
            resolved = value if value.is_absolute() else path.parent / value
            if not resolved.is_file():
                missing.append(str(resolved))
    print(path.name, 'windows=', len(rows), 'buckets=', dict(sorted(counts.items())))
    if missing:
        raise FileNotFoundError(f'{len(missing)} referenced files are missing; first: {missing[0]}')
    return rows

train_rows = inspect_manifest(TRAIN_MANIFEST)
validation_rows = inspect_manifest(VALIDATION_MANIFEST)

# Decode one real sample before spending GPU time.
output_indices = MiohRestorerV5.quality().config.output_indices if VARIANT == 'q' else MiohRestorerV5.shipping().config.output_indices
sample_dataset = MiohRestorerV5NativeDataset(VALIDATION_MANIFEST, output_indices=output_indices, degrade=False, horizontal_flip=False, time_reverse=False, deterministic=True)
sample = sample_dataset[0]
print('Sample:', sample['name'])
print('Input:', tuple(sample['inputs'].shape), 'Target:', tuple(sample['targets'].shape), 'Mask:', tuple(sample['masks'].shape))

## 5. Measure before training

Run a short synthetic CUDA profile first. Start with the buckets that actually occur in the manifest. This excludes video decoding and teacher costs, but immediately exposes an impossible batch/VRAM combination.

In [ ]:
available_sizes = sorted({int(row['bucket']) for row in train_rows})
profile_sizes = ','.join(str(size) for size in available_sizes)
PROFILE_REPORT = DRIVE_PROJECT / 'profiles' / f'v5-{VARIANT}-cuda.json'
PROFILE_REPORT.parent.mkdir(parents=True, exist_ok=True)
profile_command = [
    sys.executable, 'scripts/training/profile-mioh-restorer-v5-training.py',
    '--variant', VARIANT, '--sizes', profile_sizes, '--steps', '10',
    '--warmup', '2', '--device', 'cuda', '--output', str(PROFILE_REPORT),
]
print(' '.join(profile_command))
subprocess.run(profile_command, check=True)
print(PROFILE_REPORT.read_text(encoding='utf-8'))

## 6. Train or resume

The runner automatically resumes the current stage when its `latest.pth` exists. To start Stage 4, change `STAGE = 4`, choose a new `RUN_NAME`, and set `INITIALIZE_FROM` to the completed Stage 3 EMA checkpoint. Do not use `--resume` to cross stage boundaries.

In [ ]:
# Optional parent checkpoint for a NEW stage. Example:
# INITIALIZE_FROM = DRIVE_PROJECT / 'runs/v5-q-stage3/mioh-v5-q-stage3-latest.pth'
INITIALIZE_FROM = None

WORK_DIR.mkdir(parents=True, exist_ok=True)
latest = WORK_DIR / f'mioh-v5-{VARIANT}-stage{STAGE}-latest.pth'
command = [
    sys.executable, '-u', 'scripts/training/train-mioh-restorer-v5-colab.py',
    '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST),
    '--work-dir', str(WORK_DIR),
    '--variant', VARIANT, '--stage', str(STAGE), '--steps', str(STEPS),
    '--batch-size', str(BATCH_SIZE), '--accumulate', str(ACCUMULATE),
    '--workers', str(WORKERS), '--amp', 'auto',
    '--save-every', str(SAVE_EVERY),
    '--validate-every', str(VALIDATE_EVERY),
    '--validation-batches', str(VALIDATION_BATCHES),
]
if latest.is_file():
    command += ['--resume', str(latest)]
elif INITIALIZE_FROM is not None:
    command += ['--initialize-from', str(INITIALIZE_FROM)]

print(' '.join(command))
subprocess.run(command, check=True)

## 7. Inspect the learning curve

Lower ROI MAE and higher ROI PSNR are better. `identity_source_mae` is the unprocessed input baseline. A useful model must beat that baseline on held-out validation data; a low training loss alone is not an acceptance result.

In [ ]:
import matplotlib.pyplot as plt

metrics_path = WORK_DIR / 'metrics.jsonl'
records = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
train = [row for row in records if row.get('event') == 'train']
validation = [row for row in records if row.get('event') == 'validation']
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if train:
    axes[0].plot([r['step'] for r in train], [r['total'] for r in train])
    axes[0].set(title='Training loss', xlabel='optimizer step', ylabel='loss')
if validation:
    axes[1].plot([r['step'] for r in validation], [r['roi_psnr'] for r in validation], marker='o', label='V5 EMA')
    axes[1].set(title='Validation ROI PSNR', xlabel='optimizer step', ylabel='dB')
    axes[1].legend()
plt.show()
print('Latest checkpoint:', latest)

## 8. Return to the Mac

The `*-latest.pth` file contains raw weights, EMA weights, optimizer state, AMP scaler state, stage and local step. Keep it on Drive for resumption. After a stage passes fixed validation and visual A/B evaluation, download the checkpoint to the Mac. Export **EMA weights** there, then perform Core AI conversion, Xcode 27 `.aimodelc` compilation, numerical comparison and M1/M5 latency tests.

Do not call a Stage 3/4 pilot the final V5 model. The remaining alignment-teacher and flow-aligned temporal stages must be implemented and evaluated before V5 can replace BasicVSR++.